# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HemapavaniDontula/flyrank-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

### Distribution notes

The key numeric fields show different ranges and some skew/heavy-tail behavior.
I inspect search volume, impressions, clicks, and content age before testing
specific signals. These are descriptive observations from the supplied dataset.

In [2]:
from pathlib import Path
import os

repo_path = Path("/content/flyrank-ml")

if not repo_path.exists():
    !git clone --depth 1 https://github.com/HemapavaniDontula/flyrank-ml.git /content/flyrank-ml

os.chdir(repo_path)

print("Current directory:", os.getcwd())

Cloning into '/content/flyrank-ml'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 91 (delta 10), reused 62 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 1.89 MiB | 7.87 MiB/s, done.
Resolving deltas: 100% (10/10), done.
Current directory: /content/flyrank-ml


In [3]:
import pandas as pd
import numpy as np

data_path = Path("/content/flyrank-ml/data/raw/content_refresh_anonymized.csv")

if not data_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {data_path}"
    )

df = pd.read_csv(data_path)

print("Dataset loaded successfully")
print("Shape:", df.shape)
print("\nFirst 10 columns:")
print(df.columns.tolist()[:10])

Dataset loaded successfully
Shape: (30000, 44)

First 10 columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count']


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from pathlib import Path

data_path = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [5]:
# Section 1: Key distributions

distribution_cols = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

distribution_summary = df[distribution_cols].describe(
    percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
).T

print(distribution_summary.to_string())

                          count         mean           std   min    25%     50%      75%       90%       95%        99%       max
search_volume           27532.0   158.882391   1518.270825   0.0    0.0   10.00    20.00    110.00    390.00   2900.000   74000.0
impressions_90d         30000.0  5200.366300  16838.019547   1.0   81.0  731.00  3615.25  12136.40  22996.50  73505.830  517715.0
clicks_90d              30000.0    16.097333     75.076958   0.0    0.0    1.00     7.00     32.00     69.05    253.010    4178.0
content_age_days        30000.0   256.167800    132.707930  90.0  132.0  236.00   333.00    463.00    487.00    537.000     564.0
days_since_last_update  30000.0    46.098300     42.078709   1.0   20.0   20.00   104.00    104.00    104.00    106.000     373.0
ctr                     30000.0     0.510733      3.279162   0.0    0.0    0.07     0.29      0.65      1.09      8.330     100.0
avg_position            30000.0    16.342380     15.216790   0.0    6.2   10.80    22.30  

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*


### Signal test #1 — Staleness

**Verdict: CONFIRMED**

The staler bucket shows a more negative mean observed trend than the fresher
bucket (-14.88% versus -0.90%). This gives directional support to the
staleness signal. However, the middle bucket is positive (+6.46%), so the
relationship is not monotonic and should be treated as an investigation signal,
not a causal rule.

In [9]:
# Signal test #2: CTR versus position

ctr_test = df[
    ["position_tier", "ctr"]
].copy()

ctr_test["ctr"] = pd.to_numeric(
    ctr_test["ctr"],
    errors="coerce"
)

ctr_table = (
    ctr_test
    .groupby("position_tier", dropna=False)
    .agg(
        n=("ctr", "size"),
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

print(ctr_table.to_string(index=False))

position_tier     n  mean_ctr  median_ctr
         deep  1319  0.150212        0.00
       page_1 11814  0.652467        0.16
     page_3_5  7242  0.222484        0.03
     striking  7304  0.323239        0.11
        top_3  2321  1.483611        0.00


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Flag-linked test — Refresh / staleness

The refresh-oriented flag assumes that older content is a useful signal for
investigation. I test that assumption using days_since_last_update and the
observed trend metric.

The result is treated as directional evidence only. A positive association
would support using staleness as an investigation signal, while a mixed result
would mean the rule should be treated cautiously.

### Flag-linked test — Refresh / staleness

The refresh-oriented flag assumes that older content is useful for
investigation. The observed data supports that assumption directionally:
the staler bucket has a more negative mean trend than the fresher bucket.
However, the middle bucket is positive, so the relationship is not
monotonic. This supports using staleness as a decision-support signal,
not as a causal rule.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Flag-linked test
# Refresh/staleness flag assumption

flag_test = df[
    ["days_since_last_update", "trend_pct"]
].copy()

flag_test["days_since_last_update"] = pd.to_numeric(
    flag_test["days_since_last_update"],
    errors="coerce"
)

flag_test["trend_pct"] = pd.to_numeric(
    flag_test["trend_pct"],
    errors="coerce"
)

# Use the same 3 stable buckets as Signal #1
flag_test["staleness_bucket"] = pd.qcut(
    flag_test["days_since_last_update"],
    q=3,
    labels=["Fresher", "Middle", "Staler"]
)

flag_table = (
    flag_test
    .groupby("staleness_bucket", observed=False)
    .agg(
        n=("trend_pct", "size"),
        mean_days_since_update=("days_since_last_update", "mean"),
        mean_trend_pct=("trend_pct", "mean"),
        median_trend_pct=("trend_pct", "median")
    )
    .reset_index()
)

print(flag_table.to_string(index=False))

staleness_bucket     n  mean_days_since_update  mean_trend_pct  median_trend_pct
         Fresher 15866               17.239317       -0.904195             -41.4
          Middle  4148               22.456847        6.463772             -10.3
          Staler  9986              101.770379      -14.876842             -35.5


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*
### Practical meaning

The audit uses observed data to identify which signals are useful for
investigation rather than treating them as causal rules. The strongest
supported signals can be used for decision-support, while mixed or weak
signals should be treated cautiously and reviewed with additional context.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.